# ambit-gate: AMBIT flash protocol with software-gated Felix lifetimes

Runs a PAM-style protocol on the AMBIT (dark, saturating actinic flash, relaxation)
while the Felix TDC records contiguous lifetime windows. There is no gate cable: the
SPAD saturates during the flash, which makes the flash edges visible in the Felix
window totals, and those edges are used to align the two clocks. Windows overlapping
the flash are discarded.

Wiring: laser SYNC OUT 1 -> Felix SYNC; SPAD SMA -> Felix CH1; AMBIT on a USB-serial
adapter (115200 8N1). Close TDC-Software before running (one program can open the Felix).

All logic lives in `ambit_gate.py` next to this notebook; the cells below only call it.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))      # run the notebook from scripts/ambit-gate
import importlib, ambit_gate as ag
ag = importlib.reload(ag)

PORT = None            # e.g. "COM5" to skip the port scan
LASER_HZ = 80e6        # laser mode selector (5/10/20/40/80 MHz)
THRESHOLD_V = 1.2      # both signals are 2.5 V LVTTL
WINDOW_S = 0.1         # Felix window length

## 1. Detect the AMBIT

In [ ]:
try:
    ambit.close()          # re-running must not leave the port locked
except NameError:
    pass
ambit = ag.find_ambit(PORT)
print(f"AMBIT '{ambit.name}' on {ambit.port}, firmware {ambit.fw or '(unreported)'}")

## 2. Detect the Felix and check the count rates

SYNC should sit at the laser repetition rate. The SPAD (ch1) must be well below 1 Mcps in the dark phase; if it is at 3 Mcps with the room light on, the filter stack is not enough (see the FLIM-Lab readme).

In [ ]:
try:
    felix.close()
except NameError:
    pass
felix = ag.FelixLink().open()
felix.set_thresholds(THRESHOLD_V)
rates = felix.count_rates(0.5)
for ch, r in rates.items():
    print(f"  ch{ch} ({'SYNC' if ch == 0 else 'CH' + str(ch)}): {r:,.0f} /s")

## 3. Protocol

One dict per `arrun` line: `type` 2 = no 730 nm reference pulses, `pulses` and `freq`
set the segment length, `actinic` is the LED current in mA (0 = off, 250 = the
saturating flash). The default is 1 s dark, 1 s at 250 mA, 3 s relaxation, all at 10 Hz.

In [ ]:
PROTOCOL = [
    dict(type=2, far_red=0, pulses=5, freq=1, actinic=5, subsampling=0),   # 1 s flash
]
print(ag.build_arrun(PROTOCOL).strip())
print(f"{ag.protocol_duration(PROTOCOL):.1f} s of protocol, "
      f"{int(np.ceil((ag.PRE_S + ag.protocol_duration(PROTOCOL) + ag.POST_S) / WINDOW_S))} Felix windows")

## 3b. Run the protocol on the AMBIT only (optional)

Sends `PROTOCOL` to the AMBIT without recording the Felix, so the Felix does not need to be
connected or open. It reuses the AMBIT from step 1 if that port is still open, otherwise it
looks for the AMBIT itself. Nothing is saved; the trace is kept in `ambit_only`.

In [ ]:
try:
    if not ambit.ser.is_open:
        raise NameError
except NameError:
    ambit = ag.find_ambit(PORT)    # step 1 not run, or the port was closed

cmd = ag.build_arrun(PROTOCOL)
print("AMBIT: " + cmd.strip())
ambit_only, t_send, t_done = ag.run_arrun(ambit, cmd, PROTOCOL)


## 4. Measure

The Felix starts first, then the `arrun` command is sent. Results are saved under `runs/<timestamp>/` (AMBIT trace, every Felix window, summed decays, run.json with the alignment).

In [ ]:
out_dir = Path("runs") / time.strftime("%Y%m%d_%H%M%S")
res = ag.run_gated(ambit, felix, PROTOCOL, window_s=WINDOW_S, laser_hz=LASER_HZ,
                   threshold_v=THRESHOLD_V, out_dir=out_dir)

## 5. Plots

Top: AMBIT fluorescence ratio. Second: Felix counts per window, orange band = flash (red bars are the saturated windows, grey ones straddle an edge and are excluded). Third: lifetime proxy per window. Bottom: summed decays before and after the flash.

In [ ]:
fig = ag.plot_run(res, show=True, save_to=out_dir / "summary.png")

## 6. Per-window table

In [ ]:
print(f"{'k':>3} {'t0 [s]':>8} {'phase':>8} {'counts':>9} {'tau_mean':>9} {'tau_fit':>8}")
for r in res.summary:
    print(f"{r['k']:3d} {r['t_start_s']:8.3f} {r['phase']:>8} {r['total']:9d} "
          f"{r['tau_mean_ns']:9.3f} {r['tau_fit_ns']:8.3f}")

## 7. Fit the summed decays with the flimgui fitter (optional)

In [ ]:
from flimgui.fitting import fit_decay
period_ns = 1e9 / LASER_HZ
for phase in ("pre", "post"):
    dec = ag.summed_decay(res, phase).astype(float)
    fr = fit_decay(dec, res.bin_s * 1e9, n_exp=2, period_ns=period_ns)
    print(f"{phase}: taus {np.round(fr.taus_ns, 3)} ns, fractions {np.round(fr.fractions, 2)}, "
          f"tau_int {fr.tau_int_ns:.3f} ns, chi2r {fr.chi2r:.2f}, {fr.mode}, {fr.message}")

## 8. Close both instruments

In [ ]:
felix.close()
ambit.close()